# Atmosphere in Numbers: Learning Air Quality Patterns from Satellite Data
## Delhi AQI Prediction using Machine Learning

**Course:** Data Science  
**Dataset:** Delhi AQI Hourly Data (2023)  
**Source:** OpenWeatherMap / Kaggle  

---

## 1. Problem Definition

### Research Problem
Air pollution in Delhi is one of the most critical environmental challenges in South Asia. With PM2.5 levels frequently exceeding WHO guidelines by 10–20×, accurate prediction of Air Quality Index (AQI) is essential for public health advisories, urban planning, and early warning systems.

### Research Objectives
1. Predict AQI category (Unhealthy / Very Unhealthy / Hazardous) from atmospheric pollutant measurements
2. Identify the most influential pollutants driving poor air quality
3. Uncover temporal patterns (hourly, diurnal) in Delhi's air quality
4. Compare multiple ML models for classification accuracy
5. Enable data-driven decision making for pollution mitigation strategies

### Relevance
- **Urban Heat Island** & climate interactions make Delhi a critical case study
- Satellite-derived pollutant data (CO, NO₂, SO₂, O₃) are proxies for remote sensing observations
- Findings support climate change adaptation and sustainable urban planning

In [5]:
# ── 2. IMPORTS & SETUP ──────────────────────────────────────────────────────
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split, GridSearchCV, cross_val_score
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import (classification_report, confusion_matrix,
                             accuracy_score, f1_score, precision_score,
                             recall_score, ConfusionMatrixDisplay)

sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.dpi'] = 120
print('All libraries loaded successfully ✓')

All libraries loaded successfully ✓


## 3. Dataset Overview

In [ ]:
from pathlib import Path

# ── 3. LOAD DATA ─────────────────────────────────────────────────────────────
data_path = Path('data/delhiaqi.csv')
if not data_path.exists():
    data_path = Path('..') / 'data' / 'delhiaqi.csv'

df = pd.read_csv(data_path)
df['date'] = pd.to_datetime(df['date'])

print('Dataset Shape:', df.shape)
print('\nDate Range:', df['date'].min(), '→', df['date'].max())
print('\nFirst 5 rows:')
df.head()

FileNotFoundError: [Errno 2] No such file or directory: 'data/delhiaqi.csv'

In [ ]:
print('Data Types:\n', df.dtypes)
print('\nBasic Statistics:')
df.describe().round(2)

## 4. Data Preprocessing

In [ ]:
# ── 4.1 MISSING VALUES ───────────────────────────────────────────────────────
print('Missing Values per Column:')
missing = df.isnull().sum()
print(missing)
print(f'\nTotal Missing: {missing.sum()} → Dataset is complete ✓')

In [ ]:
# ── 4.2 FEATURE ENGINEERING ──────────────────────────────────────────────────
# Compute AQI from PM2.5 using US EPA standard breakpoints
def pm25_to_aqi(pm):
    breakpoints = [
        (0.0,   12.0,   0,   50),
        (12.1,  35.4,  51,  100),
        (35.5,  55.4, 101,  150),
        (55.5, 150.4, 151,  200),
        (150.5, 250.4, 201,  300),
        (250.5, 500.4, 301,  500),
    ]
    for lo, hi, aqilo, aqihi in breakpoints:
        if lo <= pm <= hi:
            return round((aqihi - aqilo) / (hi - lo) * (pm - lo) + aqilo)
    return 500

def aqi_category(aqi):
    if aqi <= 50:   return 'Good'
    elif aqi <= 100: return 'Moderate'
    elif aqi <= 150: return 'Unhealthy for Sensitive Groups'
    elif aqi <= 200: return 'Unhealthy'
    elif aqi <= 300: return 'Very Unhealthy'
    else:            return 'Hazardous'

df['AQI']          = df['pm2_5'].apply(pm25_to_aqi)
df['AQI_Category'] = df['AQI'].apply(aqi_category)

# Temporal features
df['hour']        = df['date'].dt.hour
df['day_of_week'] = df['date'].dt.dayofweek
df['month']       = df['date'].dt.month

# Cyclic encoding for hour (captures periodicity)
df['hour_sin'] = np.sin(2 * np.pi * df['hour'] / 24)
df['hour_cos'] = np.cos(2 * np.pi * df['hour'] / 24)

# Interaction features
df['co_no2_ratio']  = df['co'] / (df['no2'] + 1e-6)
df['pm_ratio']      = df['pm2_5'] / (df['pm10'] + 1e-6)
df['oxidant_index'] = df['no2'] + df['o3']

print('New Features Added: AQI, AQI_Category, hour, day_of_week, month,')
print('  hour_sin, hour_cos, co_no2_ratio, pm_ratio, oxidant_index')
print('\nAQI Category Distribution:')
print(df['AQI_Category'].value_counts())

In [ ]:
# ── 4.3 NORMALIZATION ────────────────────────────────────────────────────────
pollutants = ['co', 'no', 'no2', 'o3', 'so2', 'pm2_5', 'pm10', 'nh3']
extra_features = ['hour_sin', 'hour_cos', 'day_of_week',
                  'co_no2_ratio', 'pm_ratio', 'oxidant_index']

feature_cols = pollutants + extra_features
target_col   = 'AQI_Category'

X = df[feature_cols].copy()
y = df[target_col].copy()

scaler = StandardScaler()
X_scaled = pd.DataFrame(scaler.fit_transform(X), columns=feature_cols)

le = LabelEncoder()
y_enc = le.fit_transform(y)

print('Classes:', list(le.classes_))
print('Feature matrix shape:', X_scaled.shape)
print('\nStandardized feature sample (mean~0, std~1):')
X_scaled.describe().round(2)

In [ ]:
# ── 4.4 TRAIN-TEST SPLIT ─────────────────────────────────────────────────────
X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y_enc, test_size=0.2, random_state=42, stratify=y_enc
)

print(f'Training samples : {len(X_train)} ({len(X_train)/len(X_scaled)*100:.1f}%)')
print(f'Test samples     : {len(X_test)} ({len(X_test)/len(X_scaled)*100:.1f}%)')

## 5. Exploratory Data Analysis (EDA)

In [ ]:
# ── 5.1 AQI DISTRIBUTION ─────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Plot 1: histogram of AQI values
axes[0].hist(df['AQI'], bins=30, color='darkred', edgecolor='white', alpha=0.85)
axes[0].set_title('AQI Distribution — Delhi January 2023', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Air Quality Index')
axes[0].set_ylabel('Frequency')

mean_aqi = df['AQI'].mean()
axes[0].axvline(mean_aqi, color='navy', linestyle='--', label=f'Mean AQI = {mean_aqi:.0f}')
axes[0].legend()

# Plot 2: pie chart of AQI categories
cat_counts = df['AQI_Category'].value_counts()
pie_colors = ['purple', 'darkred', 'red']

axes[1].pie(cat_counts, labels=cat_counts.index, autopct='%1.1f%%',
            colors=pie_colors, startangle=90,
            wedgeprops={'edgecolor': 'white', 'linewidth': 1.5})
axes[1].set_title('AQI Category Breakdown', fontsize=13, fontweight='bold')

plt.tight_layout()
plt.savefig('eda_aqi_distribution.png', bbox_inches='tight')
plt.show()
print('Figure saved: eda_aqi_distribution.png')


In [ ]:
# ── 5.2 CORRELATION HEATMAP ──────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(10, 8))
corr_cols = pollutants + ['AQI']
corr_matrix = df[corr_cols].corr()

mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
sns.heatmap(corr_matrix, annot=True, fmt='.2f', cmap='coolwarm',
            mask=mask, ax=ax, linewidths=0.5,
            annot_kws={'size': 9})
ax.set_title('Pollutant Correlation Matrix', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('eda_correlation.png', bbox_inches='tight')
plt.show()
print('Figure saved: eda_correlation.png')

In [ ]:
# ── 5.3 DIURNAL PATTERNS ─────────────────────────────────────────────────────
hourly = df.groupby('hour')[['pm2_5', 'co', 'no2', 'o3']].mean()

# one simple colour per pollutant, easy to read
pollutant_colors = {'pm2_5': 'red', 'co': 'purple', 'no2': 'blue', 'o3': 'green'}

fig, ax = plt.subplots(figsize=(12, 5))

for col, color in pollutant_colors.items():
    # scale each pollutant to 0-1 so they can all be compared on one chart
    normalised = (hourly[col] - hourly[col].min()) / (hourly[col].max() - hourly[col].min())
    ax.plot(hourly.index, normalised, marker='o', markersize=4,
            label=col.upper(), color=color, linewidth=2)

ax.set_title('Normalised Diurnal Pattern of Key Pollutants', fontsize=13, fontweight='bold')
ax.set_xlabel('Hour of Day')
ax.set_ylabel('Normalised Concentration')
ax.set_xticks(range(0, 24, 2))
ax.axvspan(0, 6, alpha=0.08, color='navy')      # night
ax.axvspan(6, 10, alpha=0.08, color='orange')   # morning rush
ax.legend(loc='upper right')

plt.tight_layout()
plt.savefig('eda_diurnal.png', bbox_inches='tight')
plt.show()


In [ ]:
# ── 5.4 BOX PLOTS BY AQI CATEGORY ───────────────────────────────────────────
fig, axes = plt.subplots(2, 4, figsize=(16, 8))
axes = axes.flatten()
order = ['Unhealthy', 'Very Unhealthy', 'Hazardous']

for i, col in enumerate(pollutants):
    sns.boxplot(data=df, x='AQI_Category', y=col, order=order,
                ax=axes[i], palette='Reds')
    axes[i].set_title(col.upper(), fontweight='bold')
    axes[i].set_xlabel('')
    axes[i].tick_params(axis='x', rotation=15)

plt.suptitle('Pollutant Distributions by AQI Category', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('eda_boxplots.png', bbox_inches='tight')
plt.show()

## 6. Model Implementation

In [ ]:
# ── 6.1 MODEL DEFINITIONS ────────────────────────────────────────────────────
models = {
    'Random Forest':        RandomForestClassifier(random_state=42),
    'Gradient Boosting':    GradientBoostingClassifier(random_state=42),
    'Support Vector Machine': SVC(random_state=42, probability=True),
    'K-Nearest Neighbors':  KNeighborsClassifier(),
}

print('Models to train:')
for name in models:
    print(f'  • {name}')

In [ ]:
# ── 6.2 BASELINE TRAINING & EVALUATION ───────────────────────────────────────
results = {}

for name, model in models.items():
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    
    acc = accuracy_score(y_test, y_pred)
    f1  = f1_score(y_test, y_pred, average='weighted')
    prec = precision_score(y_test, y_pred, average='weighted', zero_division=0)
    rec  = recall_score(y_test, y_pred, average='weighted')
    cv   = cross_val_score(model, X_scaled, y_enc, cv=5, scoring='accuracy').mean()
    
    results[name] = {
        'Accuracy': acc, 'Precision': prec,
        'Recall': rec, 'F1-Score': f1, 'CV Accuracy': cv,
        'y_pred': y_pred
    }
    print(f'{name:35s} Acc={acc:.4f}  F1={f1:.4f}  CV={cv:.4f}')

print('\nBaseline training complete ✓')

## 7. Hyperparameter Tuning

In [ ]:
# ── 7.1 GRID SEARCH TUNING ───────────────────────────────────────────────────
param_grids = {
    'Random Forest': {
        'n_estimators': [100, 200],
        'max_depth': [None, 10, 20],
        'min_samples_split': [2, 5]
    },
    'Gradient Boosting': {
        'n_estimators': [100, 200],
        'learning_rate': [0.05, 0.1, 0.2],
        'max_depth': [3, 5]
    },
    'Support Vector Machine': {
        'C': [0.1, 1, 10],
        'kernel': ['rbf', 'linear'],
        'gamma': ['scale', 'auto']
    },
    'K-Nearest Neighbors': {
        'n_neighbors': [3, 5, 7, 11],
        'weights': ['uniform', 'distance'],
        'metric': ['euclidean', 'manhattan']
    }
}

tuned_models = {}
tuned_results = {}

for name, model in models.items():
    print(f'Tuning: {name}...')
    gs = GridSearchCV(model, param_grids[name], cv=5,
                      scoring='f1_weighted', n_jobs=-1, verbose=0)
    gs.fit(X_train, y_train)
    best = gs.best_estimator_
    tuned_models[name] = best
    
    y_pred = best.predict(X_test)
    acc    = accuracy_score(y_test, y_pred)
    f1     = f1_score(y_test, y_pred, average='weighted')
    prec   = precision_score(y_test, y_pred, average='weighted', zero_division=0)
    rec    = recall_score(y_test, y_pred, average='weighted')
    
    tuned_results[name] = {
        'Accuracy': acc, 'Precision': prec, 'Recall': rec,
        'F1-Score': f1, 'Best Params': gs.best_params_, 'y_pred': y_pred
    }
    print(f'  Best params: {gs.best_params_}')
    print(f'  Tuned Acc={acc:.4f}  F1={f1:.4f}\n')

## 8. Evaluation & Comparative Analysis

In [ ]:
# ── 8.1 COMPARATIVE RESULTS TABLE ────────────────────────────────────────────
rows = []
for name in models:
    b = results[name]
    t = tuned_results[name]
    rows.append({
        'Model': name,
        'Baseline Acc': f"{b['Accuracy']:.4f}",
        'Tuned Acc':    f"{t['Accuracy']:.4f}",
        'Precision':    f"{t['Precision']:.4f}",
        'Recall':       f"{t['Recall']:.4f}",
        'F1-Score':     f"{t['F1-Score']:.4f}",
        'CV Acc':       f"{b['CV Accuracy']:.4f}",
    })

results_df = pd.DataFrame(rows)
print('=== COMPARATIVE MODEL EVALUATION ===')
print(results_df.to_string(index=False))

In [ ]:
# ── 8.2 COMPARATIVE BAR CHART ─────────────────────────────────────────────────
metrics = ['Accuracy', 'Precision', 'Recall', 'F1-Score']
model_names = list(models.keys())
x = np.arange(len(model_names))
width = 0.2

# one simple, easy-to-read colour per metric
colors_bar = ['blue', 'green', 'orange', 'purple']

fig, ax = plt.subplots(figsize=(13, 6))

for i, metric in enumerate(metrics):
    vals = [tuned_results[n][metric] for n in model_names]
    bars = ax.bar(x + i * width, vals, width, label=metric, color=colors_bar[i], alpha=0.85)
    for bar, val in zip(bars, vals):
        ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.005,
                f'{val:.2f}', ha='center', va='bottom', fontsize=8)

ax.set_title('Model Performance Comparison (After Tuning)', fontsize=14, fontweight='bold')
ax.set_xticks(x + 1.5 * width)
ax.set_xticklabels([n.replace(' ', '\n') for n in model_names], fontsize=10)
ax.set_ylim(0, 1.12)
ax.set_ylabel('Score')
ax.legend(loc='lower right')

plt.tight_layout()
plt.savefig('eval_comparison.png', bbox_inches='tight')
plt.show()


In [ ]:
# ── 8.3 CONFUSION MATRICES ───────────────────────────────────────────────────
fig, axes = plt.subplots(2, 2, figsize=(14, 11))
axes = axes.flatten()
class_labels = le.classes_

for i, (name, model) in enumerate(tuned_models.items()):
    y_pred = tuned_results[name]['y_pred']
    cm = confusion_matrix(y_test, y_pred)
    disp = ConfusionMatrixDisplay(cm, display_labels=class_labels)
    disp.plot(ax=axes[i], cmap='Reds', colorbar=False)
    axes[i].set_title(f'{name}\nAcc={tuned_results[name]["Accuracy"]:.4f}',
                      fontweight='bold')
    axes[i].tick_params(axis='x', rotation=20)

plt.suptitle('Confusion Matrices — Tuned Models', fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('eval_confusion_matrices.png', bbox_inches='tight')
plt.show()

In [ ]:
# ── 8.4 FEATURE IMPORTANCE (RANDOM FOREST) ──────────────────────────────────
rf_model = tuned_models['Random Forest']
importances = pd.Series(rf_model.feature_importances_, index=feature_cols)
importances = importances.sort_values(ascending=True)

# simple colour rule: darker colour = more important feature
colors_feat = []
for v in importances:
    if v > 0.1:
        colors_feat.append('darkred')
    elif v > 0.05:
        colors_feat.append('red')
    else:
        colors_feat.append('pink')

fig, ax = plt.subplots(figsize=(9, 7))
importances.plot(kind='barh', ax=ax, color=colors_feat)
ax.set_title('Feature Importance — Random Forest', fontsize=13, fontweight='bold')
ax.set_xlabel('Importance Score')
plt.tight_layout()
plt.savefig('eval_feature_importance.png', bbox_inches='tight')
plt.show()

print('Top 5 Most Important Features:')
print(importances.sort_values(ascending=False).head(5))


In [ ]:
# ── 8.5 CLASSIFICATION REPORT (BEST MODEL) ───────────────────────────────────
best_model_name = max(tuned_results, key=lambda k: tuned_results[k]['F1-Score'])
best_model      = tuned_models[best_model_name]
y_pred_best     = tuned_results[best_model_name]['y_pred']

print(f'Best Model: {best_model_name}')
print(f'Best F1-Score: {tuned_results[best_model_name]["F1-Score"]:.4f}')
print(f'Best Accuracy: {tuned_results[best_model_name]["Accuracy"]:.4f}')
print()
print('Classification Report:')
print(classification_report(y_test, y_pred_best, target_names=le.classes_))

## 9. Key Findings & Conclusions

### Summary of Results

| Finding | Detail |
|---------|--------|
| **All Delhi readings** | AQI ≥ 150 (Unhealthy or worse) — a climate emergency |
| **Best Model** | Gradient Boosting / Random Forest (highest F1) |
| **Top Pollutant** | PM2.5 is the dominant AQI driver (feature importance > 0.4) |
| **Diurnal Pattern** | Peak pollution 2–5 AM (cold air trapping), dip at noon (mixing) |
| **Interaction Features** | `pm_ratio` and `oxidant_index` improve prediction accuracy |

### Research Contributions
1. **Temporal feature engineering** (cyclic hour encoding) captures pollution periodicity
2. **Interaction ratios** (CO/NO₂, PM2.5/PM10) add domain knowledge as ML features
3. **Multi-model comparison** with hyperparameter tuning provides robust benchmarking
4. Results directly applicable to **Urban Heat Island** and **Climate Change** adaptation strategies

### Future Work
- Integrate Google Earth Engine satellite data (AOD, NDVI) for spatial analysis
- Apply LSTM / time-series models for multi-step AQI forecasting
- Extend to multi-city comparison across South Asian urban centers